In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch

In [2]:
SRC=Path.cwd().absolute(); TASK=SRC.parent; ROOT=TASK.parent
INPUT=TASK/'input'; OUTPUT=TASK/'output'
CFG=json.loads((ROOT/'config.json').read_text()); SEED=CFG['seed']
np.random.seed(SEED); torch.manual_seed(SEED)

In [3]:
from fit_models import grid, fit_one, finalists

In [4]:
arrays=np.load(INPUT/'prepare'/'windows.npz')

In [ ]:
screen=pd.DataFrame([fit_one(CFG, arrays, name, 0, 'screen', OUTPUT) for name in grid(CFG)])
screen.to_csv(OUTPUT/'screen_validation.csv', index=False)
display(screen.sort_values('val_nll').head(15))

,config,replicate,val_nll,epoch,params,seconds
29,AID,0,1.231821,49,12050,61.212172
19,AA,0,1.270679,23,7394,49.313984
37,FIA,0,1.270735,26,12074,47.593536
18,AI,0,1.270798,23,7346,34.613094
35,DIA,0,1.270843,26,12050,39.684079
11,FA,0,1.271091,23,7346,33.660084
16,AD,0,1.272913,23,7322,27.220619
27,FDA,0,1.273152,26,12050,39.989045
6,DI,0,1.273671,23,7274,11.975967
14,II,0,1.273996,23,7298,18.795406


In [6]:
selected=finalists(CFG, screen)
(OUTPUT/'selection.json').write_text(json.dumps(selected, indent=2))
display(pd.DataFrame({'selected_config':selected}))

,selected_config
0,A
1,D
2,F
3,I
4,AA
5,AI
6,DD
7,FF
8,IA
9,II


In [7]:
final=pd.DataFrame([fit_one(CFG, arrays, name, rep, 'final', OUTPUT) for name in selected for rep in range(CFG['fit']['repeats'])])
final.to_csv(OUTPUT/'final_validation.csv', index=False)
display(final.groupby('config')[['val_nll','params']].agg(['mean','std']).sort_values(('val_nll','mean')))

val_nll             params     
            mean       std     mean  std
config                                  
AI      1.255926  0.025392   7346.0  0.0
AAA     1.256685  0.022780  12170.0  0.0
AA      1.260011  0.016098   7394.0  0.0
IA      1.266932  0.009513   7346.0  0.0
AID     1.271059  0.022168  12050.0  0.0
DIA     1.271085  0.002492  12050.0  0.0
FIA     1.271985  0.004604  12074.0  0.0
II      1.272272  0.006796   7298.0  0.0
FF      1.274523  0.005638   7298.0  0.0
DD      1.275824  0.005498   7250.0  0.0
III     1.278684  0.002535  12026.0  0.0
DDD     1.278704  0.003680  11954.0  0.0
AIF     1.279170  0.001711  12074.0  0.0
FFF     1.279883  0.003375  12026.0  0.0
A       1.287874  0.002556   2618.0  0.0
I       1.290410  0.005202   2570.0  0.0
F       1.290530  0.005208   2570.0  0.0
D       1.291697  0.005431   2546.0  0.0